In [1]:
import tomllib
from pathlib import Path
from dotenv import load_dotenv
from typing import TypedDict
from rich.panel import Panel
from rich.console import Console
import json
import re

In [2]:
from rich.markdown import Markdown
from rich import print as pprint
from langchain_core.messages import BaseMessage
from langchain.chat_models import init_chat_model
from langgraph.graph import (
    START,
    END,
    StateGraph,
    add_messages
)
from smolagents import CodeAgent, OpenAIModel

In [3]:
# importando as variaveis
ENV_PATH = Path('../../.env')
load_dotenv(dotenv_path=ENV_PATH)

True

In [4]:
console = Console()

In [5]:
# load configs
CONFIGS = tomllib.load(Path("config.toml").open("rb"))
MODELS = CONFIGS['models']
TEMA = CONFIGS['tema']

In [6]:
# definindo os LLMs utilizados
llm_lado_A = init_chat_model(MODELS['llm_A'])
llm_lado_B = init_chat_model(MODELS['llm_B'])
llm_jurado = init_chat_model(MODELS['llm_jurado'])

In [7]:
def extract_json(text: str):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError("Nenhum JSON encontrado no output do modelo")
    return json.loads(match.group())

In [8]:
class DebateState(TypedDict):
    topic: str
    round_: int
    history_A: list[BaseMessage]
    history_B: list[BaseMessage]
    messages: list[BaseMessage]
    scores: dict
    turn: str

In [9]:
initial_state = {
    "topic": TEMA,
    "round_": 1,
    "history_A": [],
    "history_B": [],
    "messages": [],
    "scores": {"A": 0, "B": 0},
    "turn": "A"
}

In [10]:
def agent_A(state: DebateState):
    topic = state["topic"]
    round_ = state["round_"]

    console.rule(f"[bold blue]ROUND {round_} - LADO A")

    history = "\n".join([m.content for m in state["history_A"][-3:]])

    prompt = f"""
Você é o Lado A em um debate.

Tema: {topic}

REGRAS IMPORTANTES:
- NÃO repita argumentos anteriores
- Seja estratégico e traga algo novo

Seu histórico:
{history}

Escreva UM PARÁGRAFO.
"""

    response = llm_lado_A.invoke(prompt)

    console.print(Panel(Markdown(response.content), title="🟦 Lado A"))

    return {
        "history_A": state["history_A"] + [response],
        "messages": state["messages"] + [response],
    }

In [11]:
def agent_B(state: DebateState):
    topic = state["topic"]
    round_ = state["round_"]

    console.rule(f"[bold red]ROUND {round_} - LADO B")

    history = "\n".join([m.content for m in state["history_B"][-3:]])

    prompt = f"""
Você é o Lado B em um debate.

Tema: {topic}

REGRAS IMPORTANTES:
- NÃO repita argumentos anteriores
- ataque ou responda o Lado A de forma nova

Seu histórico:
{history}

Escreva UM PARÁGRAFO.
"""

    response = llm_lado_B.invoke(prompt)

    console.print(Panel(Markdown(response.content), title="🟥 Lado B"))

    return {
        "history_B": state["history_B"] + [response],
        "messages": state["messages"] + [response],
    }

In [12]:
def judge(state: DebateState):
    round_ = state["round_"]
    topic = state["topic"]

    console.rule(f"[bold yellow]JUIZ - ROUND {round_}")

    debate_text = "\n\n".join([m.content for m in state["messages"][-2:]])

    prompt = f"""
Você é um juiz imparcial.

Responda SOMENTE JSON válido:

{{
  "winner": "A" ou "B",
  "reason": "máximo 3 frases"
}}

Tema: {topic}

Round {round_}:

{debate_text}
"""

    result = llm_jurado.invoke(prompt)

    try:
        data = extract_json(result.content)

    except Exception:
        console.print("[red]JSON inválido. Solicitando correção...[/red]")

        fix_prompt = f"""
Corrija sua resposta e retorne SOMENTE JSON válido:

{{
  "winner": "A" ou "B",
  "reason": "máximo 3 frases"
}}

Debate:
{debate_text}
"""

        corrected = llm_jurado.invoke(fix_prompt)

        try:
            data = extract_json(corrected.content)
            result = corrected
        except Exception:
            console.print("[red]Falha crítica. Usando fallback.[/red]")
            data = {"winner": "B", "reason": "Erro de parsing"}

    winner = data["winner"]
    reason = data["reason"]

    console.print(
        Panel.fit(
            f"[bold]Vencedor:[/bold] {winner}\n\n{reason}",
            title="⚖️ Decisão do Juiz"
        )
    )

    scores = state["scores"]
    scores[winner] += 1

    console.print(f"[green]Placar:[/green] {scores}")

    return {
        "messages": state["messages"] + [result],
        "scores": scores,
        "round_": round_ + 1,
        "turn": "B" if state["turn"] == "A" else "A"
    }

In [13]:
def route(state: DebateState):
    return state["turn"]

def should_continue(state: DebateState):
    return "continue" if state["round_"] <= 3 else "end"

In [14]:
def should_continue(state: DebateState):
    return "continue" if state["round_"] <= 3 else "end"

In [15]:
graph = StateGraph(DebateState)

graph.add_node("A", agent_A)
graph.add_node("B", agent_B)
graph.add_node("judge", judge)

graph.add_conditional_edges(START, route, {"A": "A", "B": "B"})

graph.add_edge("A", "B")
graph.add_edge("B", "judge")

graph.add_conditional_edges(
    "judge",
    should_continue,
    {
        "continue": "A",
        "end": END
    }
)

app = graph.compile()


In [16]:
print(app.get_graph().draw_ascii())

  +-----------+    
  | __start__ |    
  +-----------+    
      .     .      
     .      .      
    .        .     
+---+         .    
| A |*        .    
+---+ *       .    
  .    ***    .    
  .       *   .    
  .        ** .    
  .         +---+  
   .        | B |  
    ..      +---+  
      .     *      
       .   *       
        . *        
    +-------+      
    | judge |      
    +-------+      
        .          
        .          
        .          
   +---------+     
   | __end__ |     
   +---------+     


In [17]:
console.rule("[bold green]INÍCIO DO DEBATE")

console.print(
    Panel.fit(
        f"[bold yellow]Tema do debate:[/bold yellow]\n\n{TEMA}",
        title="Debate"
    )
)

result = app.invoke(initial_state)

console.print("\n[bold green]RESULTADO FINAL[/bold green]")
console.print(result["scores"])

──────────────────────────────────────────────── INÍCIO DO DEBATE ─────────────────────────────────────────────────

╭─────────────── Debate ────────────────╮
│ Tema do debate:                       │
│                                       │
│ O nome correto é bolacha ou biscoito? │
╰───────────────────────────────────────╯

──────────────────────────────────────────────── ROUND 1 - LADO A ─────────────────────────────────────────────────

╭─────────────────────────────────────────────────── 🟦 Lado A ───────────────────────────────────────────────────╮
│ O nome correto é "biscoito": além da raiz etimológica latina bis-coctus ("duas vezes cozido"), que explica a    │
│ técnica tradicional de fabricação e conecta o termo a cognatos em várias línguas (como o inglês "biscuit"),     │
│ "biscoito" funciona como termo técnico e abrangente no vocabulário alimentício — capaz de englobar variantes    │
│ doces e salgadas e de servir como padrão neutro na rotulagem e comércio nacional, enquanto "bolacha" permanece  │
│ como variedade regional coloquial que revela mais identidade local do que precisão terminológica.               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

──────────────────────────────────────────────── ROUND 1 - LADO B ─────────────────────────────────────────────────

...

JSON inválido. Solicitando correção...

╭──────────────────────────────────────────── ⚖️ Decisão do Juiz ────────────────────────────────────────────╮
│ Vencedor: B                                                                                               │
│                                                                                                           │
│ A palavra 'bolacha' é mais clara e concisa, e não faz parte da nomenclatura oficial do setor alimentício. │
╰───────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Placar: {'A': 1, 'B': 2}

RESULTADO FINAL

{'A': 1, 'B': 2}

In [18]:
pprint(result)

{
    'topic': 'O nome correto é bolacha ou biscoito?',
    'round_': 4,
    'history_A': [
        AIMessage(
            content='O nome correto é "biscoito": além da raiz etimológica latina bis-coctus ("duas vezes cozido"),
que explica a técnica tradicional de fabricação e conecta o termo a cognatos em várias línguas (como o inglês 
"biscuit"), "biscoito" funciona como termo técnico e abrangente no vocabulário alimentício — capaz de englobar 
variantes doces e salgadas e de servir como padrão neutro na rotulagem e comércio nacional, enquanto "bolacha" 
permanece como variedade regional coloquial que revela mais identidade local do que precisão terminológica.',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 834,
                    'prompt_tokens': 65,
                    'total_tokens': 899,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 704,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-Dd6hOaYDPduCZnErCospnCToXI7Zz',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--743ae7e9-5f7e-4ddc-9913-5d51be82e6bc-0',
            usage_metadata={
                'input_tokens': 65,
                'output_tokens': 834,
                'total_tokens': 899,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 704}
            }
        ),
        AIMessage(
            content='Do ponto de vista prático e comunicativo, o critério decisivo é qual forma permite 
correspondência direta e mínima ambiguidade em contextos amplos — e aí "biscoito" leva vantagem: é a palavra que 
traduz de modo franco e uniforme para termos como "biscuit"/"cookie" em traduções e comunicações internacionais, 
evita explicações regionais desnecessárias e funciona como referência estável em materiais didáticos, imprensa e 
comércio externo; optar por "biscoito" é, portanto, escolher a forma que facilita entendimento entre falantes, 
tradutores e mercados distintos, tornando‑a a designação correta quando se busca clareza e padronização.',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 2197,
                    'prompt_tokens': 186,
                    'total_tokens': 2383,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 2048,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-Dd6hloMExfa6e4o6vglqyD8NSeui8',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--1fe6fc39-f685-4581-addd-693308ede341-0',
            usage_metadata={
                'input_tokens': 186,
                'output_tokens': 2197,
                'total_tokens': 2383,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasonin